> **Note:** This notebook only reads from the CSV file for matching purposes. It never modifies or writes to the CSV file.

# Check context_quote matches in PDF
This notebook checks if each `context_quote` from the processed CSV appears in the original PDF, and highlights matches in a new PDF.

In [13]:
import pandas as pd
import fitz  # PyMuPDF
import os
import re
from rapidfuzz import fuzz  # for improved fuzzy matching

# Paths
csv_path = '/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_documents_processed/gpt4o_mini_v1_27_processed.csv'
pdf_path = '/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/New_Documents/27.pdf'
output_pdf_path = '/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Check_doc/gpt4o_mini_27_highlighted.pdf'

# Load CSV (read-only, never modify or write to the CSV)
df = pd.read_csv(csv_path)

# Open PDF
doc = fitz.open(pdf_path)

def normalize_text(text):
    # Lowercase, collapse whitespace, preserve parentheses/brackets, remove other punctuation
    text = str(text).lower().replace('\n', ' ')
    text = re.sub(r'[^a-z0-9 ()\\[\\]]+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def robust_match(quote, page_text, threshold=80, window_size=400, step=50):
    """
    Try direct substring match, then normalized substring, then sliding window fuzzy match.
    """
    # Direct substring match (raw, case-insensitive)
    if quote.lower() in page_text.lower():
        return True
    quote_norm = normalize_text(quote)
    page_norm = normalize_text(page_text)
    # Normalized substring match
    if quote_norm in page_norm:
        return True
    # Fallback to sliding window fuzzy match
    page_words = page_norm.split()
    if len(page_words) <= window_size:
        score = fuzz.partial_ratio(quote_norm, ' '.join(page_words))
        return score >= threshold
    for i in range(0, len(page_words) - window_size + 1, step):
        window = ' '.join(page_words[i:i+window_size])
        score = fuzz.partial_ratio(quote_norm, window)
        if score >= threshold:
            return True
    return False

results = []
for idx, row in enumerate(df.itertuples(index=False)):
    quote = str(row.context_quote).strip()
    found = False
    for page_num in range(len(doc)):
        page = doc[page_num]
        page_text = page.get_text()
        # Use robust match: substring first, then fuzzy
        if robust_match(quote, page_text):
            found = True
            # Try to highlight the exact quote if possible
            text_instances = page.search_for(quote)
            if not text_instances and len(quote) > 80:
                text_instances = page.search_for(quote[:80])
            for inst in text_instances:
                page.add_highlight_annot(inst)
            break
    results.append({'row': idx+2, 'context_quote': quote, 'match': found})

doc.save(output_pdf_path, garbage=4, deflate=True)

for r in results:
    print(f"Row {r['row']}: {'MATCH' if r['match'] else 'NO MATCH'}")
    if not r['match']:
        print(f"  Quote: {r['context_quote'][:120]}...")

print(f'Highlighted PDF saved to: {output_pdf_path}')

Row 2: NO MATCH
  Quote: The RPS FLiDAR Buoy will consist of instrumentation and supporting systems atop a floating moored buoy platform... The h...
Row 3: MATCH
Row 4: MATCH
Row 5: MATCH
Row 6: MATCH
Row 7: MATCH
Row 8: NO MATCH
  Quote: Vessel speeds for all project vessels shall not exceed 10 knots within the Lease Area....
Row 9: NO MATCH
  Quote: Furthermore, trenching activities are prohibited within 500 meters of the identified historic shipwreck (No. 551B) at al...
Highlighted PDF saved to: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Check_doc/gpt4o_mini_27_highlighted.pdf
